In [1]:
#install
#!pip install numpy
#!pip install scipy
#!pip install matplotlib
#!pip install mpi4py

In [2]:
#imports
import numpy as np
from scipy.linalg import expm
#from multiprocessing import Pool
import matplotlib.pyplot as plt
from matplotlib import colors

In [3]:
def base_qpd():
    c_op = np.eye(2)
    d_op = np.array([[0, 1], [-1, 0]])
    return [c_op, d_op]

def cyclic(n):
    t = 2*np.pi/n
    return([np.round(np.array([[np.cos(t*i),np.sin(t*i)],[np.sin(-t*i),np.cos(t*i)]]),7) for i in range(n)])
 
def dicyclic(n):
    x = np.array([[0,1],[-1,0]])
    c = cyclic(2*n)
    return(c+[x@m for m in c])

In [4]:
class QuantumPrisonersDilema:
    def __init__(self, EntanglementOperator: np.ndarray, strategy_space):
        self.J = EntanglementOperator
        self.strategy_space = strategy_space
        self.alice_payoff_list = np.array([3, 0, 5, 1])
        self.bob_payoff_list = np.array([3, 5, 0, 1])
        self.initial_state = np.array([1, 0, 0, 0])

    def play(self, alice_move: np.ndarray, bob_move: np.ndarray) -> np.ndarray:
        """
        Return the final state after alice and bob play their moves
        """
        fs_vect = (self.J.conj().transpose() @ np.kron(alice_move, bob_move) @ self.J) @ self.initial_state
        return fs_vect

    def _calculate_payoff(self, final_state: np.ndarray) -> tuple[float, float]:
        """
        Return a tuple of (alice payoff, bob payoff)
        """
        payoff_vect = np.abs(final_state)**2
        pay_A = np.dot(self.alice_payoff_list, payoff_vect)
        pay_B = np.dot(self.bob_payoff_list, payoff_vect)
        return (pay_A, pay_B)



    def payoff(self, alice_move: np.ndarray, bob_move: np.ndarray) -> tuple[float, float]:
        return self._calculate_payoff(self.play(alice_move, bob_move))

    def best_response(self, alice_move: np.ndarray) -> np.ndarray:
        """
        given alice_move return the best move for bob.

        Question: is best the largest difference in bobs favor? or the largest score for bob?
        """
        pass

    def find_pareto_optimums(self):
        """np.ndarray, np.ndarray)
        find a pair of moves that is pareto optimal

        A Pareto optimum is a state of resource allocation where no individual's situation can be improved without making at least one other individual worse off.
        """
        optimums = []
        Strats = self.strategy_space
        for i, s1 in enumerate(Strats):
            for j, s2 in enumerate(Strats):
                if self.is_pareto_optimal((s1,s2),(i, j)):
                    optimums.append((s1,s2))
        return optimums

    def is_pareto_optimal(self, currentstrats, indexes) -> bool:
        Strats = self.strategy_space
        initscore = self.payoff(alice_move=currentstrats[0], bob_move=currentstrats[1])
        for i, s1 in enumerate(Strats):
            for j, s2 in enumerate(Strats):
                compare = self.payoff(s1, s2)
                greaterA = compare[0] > initscore[0]
                greaterB = compare[1] > initscore[1]
                eqA =  compare[0] == initscore[0]
                eqB =  compare[1] == initscore[1]
                if (eqA and greaterB) or (eqB and greaterA):
                    return False
                notsameck = (i != indexes[0] or j != indexes[1])
                if greaterA and greaterB and notsameck:
                    return False
        return True

    def find_nash_equilibrium(self):
        """
        find a pair of moves the is a nash equilibrium

        A Nash equilibrium is a set of strategies where no player can improve their payoff by unilaterally changing their own strategy, assuming all other players' strategies remain constant.
        """
        equilibriums = []
        Strats = self.strategy_space
        for s1 in Strats:
            for s2 in Strats:
                if self.is_nash_equilibrium((s1, s2)):
                    equilibriums.append((s1, s2))
        return equilibriums

    def is_nash_equilibrium(self, currentstrats):
        Strats = self.strategy_space
        initscore = self.payoff(alice_move=currentstrats[0], bob_move=currentstrats[1])
        for s1 in Strats:
            compare = self.payoff(s1, currentstrats[1])
            greatereqA = round(compare[0], 14) > round(initscore[0], 14)
            if greatereqA:
                return False

        for s2 in Strats:
            compare = self.payoff(currentstrats[0], s2)
            greatereqB = round(compare[1], 14) > round(initscore[1], 14)
            if greatereqB:
                return False
        return True

    def get_all_scores(self):
        all_alice_scores = []
        all_bob_scores = []
        Strats = self.strategy_space
        for s1 in Strats:
            new_row_alice = []
            new_row_bob = []
            for s2 in Strats:
                new_row_alice.append(self.payoff(s1, s2)[0])
                new_row_bob.append(self.payoff(s1, s2)[1])
            all_alice_scores.append(new_row_alice)
            all_bob_scores.append(new_row_bob)
        return (np.array(np.round(all_alice_scores, 2)), np.array(np.round(all_bob_scores, 2)))

    def plot(self, player):
        """
        If the self.strategy_space is discrete then plot a payoff matrix for all moves.

        If not, maybe something related to a cayleigh graph?
        """
        strats = self.strategy_space
        if player == 'alice':
            payoff_matrix = np.array([[self.payoff(alice_strat, bob_strat)[
            0] for alice_strat in strats] for bob_strat in strats])
        if player ==  'bob':
            payoff_matrix = np.array([[self.payoff(bob_strat, alice_strat)[
                                         0] for alice_strat in strats] for bob_strat in strats])
        plt.imshow(payoff_matrix)
        plt.colorbar()
        plt.show()

In [5]:
def J(a, b, g) -> np.ndarray:
    j_var = np.round(np.array([[(np.e**(1j*g) + np.e**(1j*a*g) + np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g))/4,
  (1j/4)*(np.e**(1j*g) - np.e**(1j*a*g) + np.e**(1j*b*g) - np.e**(1j*(1 + a + b)*g)),
  (1j/4)*(np.e**(1j*g) + np.e**(1j*a*g) - np.e**(1j*b*g) - np.e**(1j*(1 + a + b)*g)),
  (-np.e**(1j*g) + np.e**(1j*a*g) + np.e**(1j*b*g) - np.e**(1j*(1 + a + b)*g))/4],
 [(1j/4)*(-np.e**(1j*g) + np.e**(1j*a*g) - np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g)),
  (np.e**(1j*g) + np.e**(1j*a*g) + np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g))/4,
  (np.e**(1j*g) - np.e**(1j*a*g) - np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g))/4,
  (1j/4)*(np.e**(1j*g) + np.e**(1j*a*g) - np.e**(1j*b*g) - np.e**(1j*(1 + a + b)*g))],
 [(1j/4)*(-np.e**(1j*g) - np.e**(1j*a*g) + np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g)),
  (np.e**(1j*g) - np.e**(1j*a*g) - np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g))/4,
  (np.e**(1j*g) + np.e**(1j*a*g) + np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g))/4,
  (1j/4)*(np.e**(1j*g) - np.e**(1j*a*g) + np.e**(1j*b*g) - np.e**(1j*(1 + a + b)*g))],
 [(-np.e**(1j*g) + np.e**(1j*a*g) + np.e**(1j*b*g) - np.e**(1j*(1 + a + b)*g))/4,
  (1j/4)*(-np.e**(1j*g) - np.e**(1j*a*g) + np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g)),
  (1j/4)*(-np.e**(1j*g) + np.e**(1j*a*g) - np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g)),
  (np.e**(1j*g) + np.e**(1j*a*g) + np.e**(1j*b*g) + np.e**(1j*(1 + a + b)*g))/4]]),7)
    return j_var

In [6]:
def find_best_ab(num_steps, start, stop, strats):
    best_ab = 0.0
    best_gamma = 0.0
    best_scores = (0, 0)
    ab_range = np.linspace(start, stop, num_steps, True, True)
    gamma_range = np.linspace(0, np.pi/2, num_steps, True, True)
    for AB in ab_range[0]: #A and B
            for g in gamma_range[0]:
                    qpd = QuantumPrisonersDilema(J(AB, AB, g), strats)
                    nash_lst = qpd.find_nash_equilibrium()
                    if len(nash_lst) == 1:
                        scores = qpd.payoff(nash_lst[0][0], nash_lst[0][1])
                        if scores[0] > best_scores[0] or scores[1] > best_scores[1]:
                            best_scores = scores
                            best_ab = AB
                            best_gamma = g

    return best_ab, best_gamma, best_scores

In [ ]:
print(find_best_ab(100, 0, 10, dicyclic(4)))